# 📈 Stock Market Prediction – Hybrid CNN + LSTM
**End-to-end pipeline with candlestick pattern detection**

Run each cell in order. GPU runtime recommended (Runtime → Change runtime type → T4 GPU).

## Step 1 · Install dependencies

In [ ]:
%%capture
!pip install yfinance tensorflow scikit-learn matplotlib plotly streamlit pyngrok -q

## Step 2 · Clone / upload project files

In [ ]:
# Option A: If you uploaded a ZIP to Colab
# !unzip stock_prediction.zip -d stock_prediction
# %cd stock_prediction

# Option B: Write files inline (paste each .py file below)
import os
os.makedirs('stock_prediction', exist_ok=True)
%cd stock_prediction
print('Working directory:', os.getcwd())

## Step 3 · Quick data check

In [ ]:
from data_loader import download_all, TICKERS
datasets = download_all()
for t, df in datasets.items():
    print(f'{t:6s}: {len(df):5d} rows  {df.index[0].date()} → {df.index[-1].date()}')

## Step 4 · Candlestick pattern summary

In [ ]:
from pattern_detection import get_pattern_summary, describe_latest_patterns

ticker = 'AAPL'
df     = datasets[ticker]

print(f'Pattern frequency for {ticker}:')
print(get_pattern_summary(df).to_string())

print('\nLatest-day patterns:')
info = describe_latest_patterns(df)
print(f"  Date  : {info['date']}")
print(f"  Fired : {info['fired']}")

## Step 5 · Preprocessing smoke-test

In [ ]:
from preprocessing import prepare_data
splits, scaler = prepare_data(df, window=30)

for split_name, sp in splits.items():
    print(f"{split_name:5s}: ohlcv={sp['ohlcv'].shape}  patterns={sp['patterns'].shape}  y_price={sp['price'].shape}")

## Step 6 · Train all models (AAPL, 50 epochs for demo)

In [ ]:
# Adjust epochs to 100+ for full training
from train import main as train_main
train_main(ticker='AAPL', window=30, epochs=50, batch=32)

## Step 7 · Train additional tickers (optional)

In [ ]:
# Uncomment tickers you want to train
from train import main as train_main

# for ticker in ['MSFT', 'TSLA', 'AMZN', 'GOOGL']:
#     train_main(ticker=ticker, window=30, epochs=50, batch=32)

## Step 8 · Launch Streamlit app with ngrok tunnel

In [ ]:
# 1. Get a free token from https://dashboard.ngrok.com/
NGROK_TOKEN = 'YOUR_NGROK_TOKEN_HERE'   # ← paste your token

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN

# Kill any existing tunnels
ngrok.kill()

# Start Streamlit in the background
import subprocess, threading
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.headless', 'true'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

import time; time.sleep(3)   # wait for server

tunnel = ngrok.connect(8501)
print('\n🚀 Streamlit app URL:', tunnel.public_url)

## Step 9 · Review saved outputs
Plots and metrics are saved under `outputs/<TICKER>/`.

In [ ]:
import matplotlib.pyplot as plt, os

ticker = 'AAPL'
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, img in zip(axes, ['actual_vs_predicted.png',
                           'confusion_matrix.png',
                           'model_comparison.png']):
    path = f'outputs/{ticker}/{img}'
    if os.path.exists(path):
        ax.imshow(plt.imread(path))
        ax.axis('off')
        ax.set_title(img.replace('_', ' ').replace('.png',''))

plt.tight_layout()
plt.show()